<a href="https://colab.research.google.com/github/Safin44/my-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Safin44/my-flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** Search performance / CTR opportunity ranking. The question is whether a modest, interpretable model can rank content items whose clicks decline in the following month, using only signals available at the March 2026 decision cutoff.

**Method:** Logistic Regression with median imputation and standardization. It is an interpretable, regularized baseline for a binary directional proxy label. We will inspect permutation importance and false positives/negatives rather than assume complexity means quality.

**Target:** April clicks at least 20% below March clicks, among items with positive March clicks. This is a retrospective evaluation label only; April values never enter the feature matrix. It is a proxy, not proof a page needs editing.

**Features:** March impressions, clicks, CTR, mean position, and sessions if present. IDs are used only for grouping/joins and never as predictors.

In [1]:
# Setup and data contract. Run in Colab with an authorized Hugging Face READ token
# saved as the HF_TOKEN secret; locally, use the HF_TOKEN environment variable.
%pip -q install duckdb pandas numpy scikit-learn huggingface_hub

import os, warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score
from sklearn.inspection import permutation_importance
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN missing. Accept FlyRank/internship-warehouse access terms and configure an authorized READ token as Colab secret HF_TOKEN.")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
DECISION_MONTH = "2026-03"

schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FACT}', hive_partitioning=true) LIMIT 0"
).df()
columns = {c.lower() for c in schema["column_name"]}

def pick(candidates, required=True):
    for c in candidates:
        if c.lower() in columns:
            return c.lower()
    if required:
        raise KeyError(f"Required field not found. Tried {candidates}. Available fields: {sorted(columns)}")
    return None

date_col = pick(["report_date", "date"])
client_col = pick(["client_hash_id"])
content_col = pick(["content_hash_id"])
impr_col = pick(["gsc_impressions", "impressions"])
click_col = pick(["gsc_clicks", "clicks"])
pos_col = pick(["gsc_avg_position", "gsc_position", "average_position", "position"])
sessions_col = pick(["ga4_sessions", "sessions"], required=False)
availability_col = pick(["gsc_available", "gsc_is_available", "has_gsc_data",
                         "search_console_available", "is_gsc_available", "gsc_data_available"])

session_expr = f"SUM(COALESCE({sessions_col}, 0))::DOUBLE" if sessions_col else "NULL::DOUBLE"
def monthly_sql(start, end, prefix):
    return f'''
    SELECT {client_col} AS client_id, {content_col} AS content_id,
           SUM(COALESCE({impr_col}, 0))::DOUBLE AS {prefix}_impressions,
           SUM(COALESCE({click_col}, 0))::DOUBLE AS {prefix}_clicks,
           CASE WHEN SUM(COALESCE({impr_col}, 0)) > 0
                THEN SUM(COALESCE({click_col}, 0))::DOUBLE / SUM(COALESCE({impr_col}, 0))
                ELSE NULL END AS {prefix}_ctr,
           AVG(TRY_CAST({pos_col} AS DOUBLE))::DOUBLE AS {prefix}_position,
           {session_expr} AS {prefix}_sessions
    FROM read_parquet('{FACT}', hive_partitioning=true)
    WHERE CAST({date_col} AS DATE) >= DATE '{start}'
      AND CAST({date_col} AS DATE) < DATE '{end}'
      AND {availability_col} IS TRUE
    GROUP BY {client_col}, {content_col}
    '''
march = monthly_sql("2026-03-01", "2026-04-01", "march")
april = monthly_sql("2026-04-01", "2026-05-01", "april")
df = con.execute(f'''
WITH m AS ({march}), a AS ({april})
SELECT m.client_id, m.content_id, m.march_impressions AS impressions,
       m.march_clicks AS clicks, m.march_ctr AS ctr,
       m.march_position AS mean_position, m.march_sessions AS sessions,
       CASE WHEN m.march_clicks > 0 AND COALESCE(a.april_clicks, 0) <= 0.8*m.march_clicks
            THEN 1 ELSE 0 END AS decline_label
FROM m LEFT JOIN a USING (client_id, content_id)
WHERE m.march_impressions > 0 AND m.march_clicks > 0
''').df()

if df.empty:
    raise RuntimeError("No eligible March rows after availability and volume filters; inspect schema/access.")
df["ctr"] = pd.to_numeric(df["ctr"], errors="coerce")
df["mean_position"] = pd.to_numeric(df["mean_position"], errors="coerce")
df["impressions"] = pd.to_numeric(df["impressions"], errors="coerce")
df["clicks"] = pd.to_numeric(df["clicks"], errors="coerce")
df["sessions"] = pd.to_numeric(df["sessions"], errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["client_id", "content_id"])
# IDs are only keys; no ID is included in feature_cols.
feature_cols = ["impressions", "clicks", "ctr", "mean_position"]
if df["sessions"].notna().any():
    feature_cols.append("sessions")
df = df.dropna(subset=["decline_label"]).reset_index(drop=True)
df["decline_label"] = df["decline_label"].astype(int)
print("Decision snapshot:", DECISION_MONTH)
print("Rows:", len(df), "| clients:", df["client_id"].nunique(),
      "| proxy-positive prevalence:", round(df["decline_label"].mean(), 4))
print("Features:", feature_cols)
display(df[feature_cols + ["decline_label"]].describe().T)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision snapshot: 2026-03
Rows: 68837 | clients: 44 | proxy-positive prevalence: 0.6145
Features: ['impressions', 'clicks', 'ctr', 'mean_position', 'sessions']


,count,mean,std,min,25%,50%,75%,max
impressions,68837.0,3758.018987,8209.704320,1.000000,481.000000,1436.000000,3946.000000,617124.0
clicks,68837.0,11.938812,41.790389,1.000000,1.000000,3.000000,10.000000,5668.0
ctr,68837.0,0.011795,0.059798,0.000007,0.001552,0.003159,0.006289,1.0
mean_position,68837.0,10.996911,10.280361,0.000000,4.308910,7.107100,14.073159,257.0
sessions,68837.0,15.691881,51.086191,0.000000,0.000000,2.000000,10.000000,2603.0
decline_label,68837.0,0.614481,0.486721,0.000000,0.000000,1.000000,1.000000,1.0


## 2. Split design

Use a **grouped client holdout**: every content item belonging to a client stays entirely in either train or test. This reduces leakage from client-specific patterns across the split. The random split is reproducible (`random_state=42`).

March features form the decision-time snapshot. April is used only to construct the retrospective evaluation label, never as a predictor. We compare the model and the hand-written baseline on exactly the same test rows and the same metrics. If the dataset has too few clients or only one target class in the holdout, the notebook stops with a clear diagnostic instead of reporting misleading metrics.

In [2]:
# Grouped split: client IDs are split metadata, not model features.
if df["client_id"].nunique() < 3:
    raise ValueError("Need at least 3 distinct clients for a meaningful grouped train/test split.")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(df, y=df["decline_label"], groups=df["client_id"]))
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()
X_train, X_test = train[feature_cols], test[feature_cols]
y_train, y_test = train["decline_label"], test["decline_label"]
if y_train.nunique() < 2 or y_test.nunique() < 2:
    raise ValueError("Grouped split has only one class in train or test. Adjust split strategy and document it; do not report ROC-AUC for a single-class test set.")
print(f"Train rows={len(train):,}; test rows={len(test):,}; "
      f"train clients={train.client_id.nunique():,}; test clients={test.client_id.nunique():,}")
print("Client overlap (must be 0):", len(set(train.client_id) & set(test.client_id)))
assert len(set(train.client_id) & set(test.client_id)) == 0

Train rows=62,798; test rows=6,039; train clients=33; test clients=11
Client overlap (must be 0): 0


## 3. Train + compare vs my baseline

The model is a median-imputed, standardized Logistic Regression with balanced class weights. The baseline is a transparent **CTR opportunity heuristic**: within position bands, compare an item's CTR with the median CTR of comparable-position items; prioritize positive CTR gaps weighted by log impressions. This is the Week-4-style hand-written rule reconstructed from the same March features if the Week-4 generated CSV is not present. No outcome label or April feature is used to calculate either score.

Metrics use the same grouped test rows:
- **Average precision (AP):** primary ranking metric for the imbalanced decline proxy.
- **Precision@10:** fraction of the ten highest-ranked test items that have the proxy label.
- **ROC-AUC:** secondary discrimination metric.

These metrics assess ranking against a noisy directional proxy, not causal impact or guaranteed SEO wins.

In [3]:
# Train the interpretable model.
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, random_state=42))
])
model.fit(X_train, y_train)
model_score = model.predict_proba(X_test)[:, 1]

# Prefer the exact Week-4 generated queue if it exists and exposes join keys + score.
# The generated CSV is intentionally git-ignored, so a clean clone may not have it.
baseline_score = None
w04_path = OUT_DIR / "baseline_action_score.csv"
if w04_path.exists():
    try:
        w04 = pd.read_csv(w04_path)
        key_aliases = {
            "client": ["client_id", "client_hash_id", "client_key"],
            "content": ["content_id", "content_hash_id", "content_key"],
            "score": ["action_score", "baseline_score", "score", "priority_score"]
        }
        def first_existing(candidates, frame):
            return next((c for c in candidates if c in frame.columns), None)
        wc = first_existing(key_aliases["client"], w04)
        wk = first_existing(key_aliases["content"], w04)
        ws = first_existing(key_aliases["score"], w04)
        if wc and wk and ws:
            w04_map = w04[[wc, wk, ws]].drop_duplicates([wc, wk])
            aligned = test[["client_id", "content_id"]].merge(
                w04_map, left_on=["client_id", "content_id"], right_on=[wc, wk], how="left", sort=False
            )
            if aligned[ws].notna().all():
                baseline_score = pd.to_numeric(aligned[ws], errors="coerce").to_numpy()
                print("Baseline source: exact Week-4 queue", w04_path)
    except Exception as exc:
        print("Could not align Week-4 queue; using documented March-only reconstruction:", repr(exc))

if baseline_score is None:
    # Transparent Week-4-style reconstruction from March-only signals.
    # Position bands prevent comparing CTRs across very different ranking positions.
    df["position_band"] = pd.cut(
        pd.to_numeric(df["mean_position"], errors="coerce"),
        bins=[-np.inf, 3, 5, 8, 12, 20, np.inf], labels=False, include_lowest=True
    )
    band_ctr = df.groupby("position_band", dropna=False)["ctr"].median()
    test_band_id = pd.cut(
        pd.to_numeric(test["mean_position"], errors="coerce"),
        bins=[-np.inf, 3, 5, 8, 12, 20, np.inf], labels=False, include_lowest=True
    )
    expected_ctr = test_band_id.map(band_ctr).astype(float)
    expected_ctr = expected_ctr.fillna(df["ctr"].median())
    ctr_gap = (expected_ctr.to_numpy() - test["ctr"].to_numpy()).clip(min=0)
    baseline_score = ctr_gap * np.log1p(test["impressions"].to_numpy())
    print("Baseline source: reconstructed Week-4-style rule (exact queue unavailable or not alignable).")

baseline_score = np.nan_to_num(np.asarray(baseline_score, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

def precision_at_k(y, score, k=10):
    k = min(k, len(y))
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[order].mean()) if k else float("nan")

def evaluate(name, y, score):
    return {
        "method": name,
        "average_precision": average_precision_score(y, score),
        "precision_at_10": precision_at_k(y, score, 10),
        "roc_auc": roc_auc_score(y, score)
    }

comparison = pd.DataFrame([
    evaluate("Week-4-style hand-written baseline", y_test, baseline_score),
    evaluate("Logistic Regression", y_test, model_score)
])
display(comparison.round(4))
comparison.to_json(OUT_DIR / "w05_model_comparison.json", orient="records", indent=2)
print("Saved metrics receipt:", OUT_DIR / "w05_model_comparison.json")
print("Baseline rule: positive within-position CTR shortfall × log(1 + March impressions).")

Baseline source: reconstructed Week-4-style rule (exact queue unavailable or not alignable).


,method,average_precision,precision_at_10,roc_auc
0,Week-4-style hand-written baseline,0.6040,0.5,0.4497
1,Logistic Regression,0.7581,1.0,0.6444


Saved metrics receipt: work/outputs/w05_model_comparison.json
Baseline rule: positive within-position CTR shortfall × log(1 + March impressions).


## 4. Errors and interpretation

Review false positives and false negatives at a transparent 0.5 model-probability threshold. For ranking use, also inspect the top ten separately. A false positive may be normal seasonality, low sample volume, or a proxy label that is not actionable. A false negative may reflect weak signals or a real decline not predictable from March aggregates.

Permutation importance is measured on the held-out set using average precision. With correlated search metrics, importance is predictive and conditional on this dataset; it is not causal evidence. A model should only replace the baseline if the improvement is repeatable and operationally meaningful.

In [4]:
# Error review at a declared threshold; ranking scores above remain the primary comparison.
test_review = test[["client_id", "content_id", "impressions", "clicks", "ctr",
                    "mean_position", "decline_label"]].copy()
test_review["model_probability"] = model_score
test_review["baseline_score"] = baseline_score
test_review["predicted_label_at_0_5"] = (model_score >= 0.5).astype(int)
test_review["error_type"] = np.select(
    [(test_review.predicted_label_at_0_5 == 1) & (test_review.decline_label == 0),
     (test_review.predicted_label_at_0_5 == 0) & (test_review.decline_label == 1)],
    ["false_positive", "false_negative"], default="correct"
)
print("Error counts:")
display(test_review["error_type"].value_counts().rename_axis("error_type").to_frame("n"))

print("Highest-ranked ten held-out items (pseudonymous keys only):")
top10 = test_review.sort_values("model_probability", ascending=False).head(10).copy()
top10["review_note"] = np.where(
    top10.error_type.eq("false_positive"),
    "Wrong under proxy: inspect seasonality, volume and whether CTR opportunity is actionable.",
    np.where(top10.error_type.eq("false_negative"),
             "Missed proxy decline: available March signals may not capture the cause.",
             "Correct under proxy only; verify page context before action.")
)
display(top10[["client_id", "content_id", "model_probability", "baseline_score",
               "decline_label", "error_type", "review_note"]])
# Do not export IDs or row-level data. Only aggregate metrics are persisted.
top10_error_summary = top10["error_type"].value_counts().to_dict()
print("Top-10 error summary:", top10_error_summary)

try:
    perm = permutation_importance(
        model, X_test, y_test, scoring="average_precision",
        n_repeats=5, random_state=42, n_jobs=-1
    )
    importance = pd.DataFrame({
        "feature": feature_cols,
        "mean_ap_decrease": perm.importances_mean,
        "std": perm.importances_std
    }).sort_values("mean_ap_decrease", ascending=False)
    print("Held-out permutation importance (positive AP decrease means useful signal):")
    display(importance.round(5))
except Exception as e:
    print("Permutation importance could not be calculated:", repr(e))

# Aggregate error interpretation; no claims are hard-coded from unseen results.
print(f"Test proxy prevalence: {y_test.mean():.3f}")
print(f"Model AP minus baseline AP: "
      f"{comparison.loc[comparison.method.eq('Logistic Regression'),'average_precision'].iloc[0] - comparison.loc[comparison.method.str.startswith('Week-4'),'average_precision'].iloc[0]:+.4f}")
print("Interpretation rule: if model metrics do not improve, keep the simpler baseline.")

Error counts:


,n
error_type,
correct,3786
false_positive,1466
false_negative,787


Highest-ranked ten held-out items (pseudonymous keys only):


,client_id,content_id,model_probability,baseline_score,decline_label,error_type,review_note
68490,client_cd12bcfd98942aa1,content_9a5496694a583150,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68743,client_f623b01661d4bfe4,content_b589c670c71a5db2,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68790,client_f623b01661d4bfe4,content_b11dddc3f12c8b25,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
67729,client_f623b01661d4bfe4,content_507653893440072c,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68122,client_f623b01661d4bfe4,content_b3684fd034d1f273,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68784,client_cd12bcfd98942aa1,content_45b21e26eaa53525,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68833,client_f623b01661d4bfe4,content_7d9c2bb43460c94d,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68531,client_f623b01661d4bfe4,content_f4887eb4e7975c51,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68423,client_f623b01661d4bfe4,content_0536a83c99505355,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...
68540,client_f623b01661d4bfe4,content_4fa145e34d62c97a,0.999998,0.0,1,correct,Correct under proxy only; verify page context ...


Top-10 error summary: {'correct': 10}
Held-out permutation importance (positive AP decrease means useful signal):


,feature,mean_ap_decrease,std
2,ctr,0.09029,0.01070
3,mean_position,0.02840,0.00199
1,clicks,0.00962,0.00102
4,sessions,0.00221,0.00065
0,impressions,0.00007,0.00013


Test proxy prevalence: 0.628
Model AP minus baseline AP: +0.1541
Interpretation rule: if model metrics do not improve, keep the simpler baseline.


## Self-check

Before committing, run the notebook from top to bottom in the authorized environment.

- [ ] Warehouse access is approved and `HF_TOKEN` is stored as a secret/environment variable, never in the notebook.
- [ ] The notebook runs without errors and the actual outputs are saved.
- [ ] Grouped split has zero client overlap and both classes in train/test.
- [ ] The model and baseline use the same held-out rows, labels, and metrics.
- [ ] Features are March-only; April appears only in the retrospective label.
- [ ] Hash IDs are grouping keys only, never model inputs; no private URLs, titles, or queries are reconstructed.
- [ ] The model-vs-baseline table, error counts, top-ten review, and permutation importance have been inspected.
- [ ] Commit `work/notebooks/w05_model.ipynb` and `work/outputs/w05_model_comparison.json`. Do not commit row-level exports or tokens.
- [ ] Read the research paper linked on the Week 5 card; it is not included in this uploaded skeleton, so its contents are not summarized here.